 ## LLM을 활용한 데이터 전처리 실습

[Gemini API 구조화 된 출력 문서 링크](https://ai.google.dev/gemini-api/docs/structured-output?hl=ko&_gl=1*m9bi85*_up*MQ..*_ga*ODAzMTc0MDM1LjE3NDgzOTI3MTA.*_ga_P1DBVKWT6V*czE3NDgzOTI3MDkkbzEkZzAkdDE3NDgzOTI3MDkkajYwJGwwJGgxNjE4MjQyNjEz)

 **실습 목표:**

 1. 고객 문의 데이터 정제 및 분류

 2. 개인정보 자동 마스킹 및 익명화

 3. 설문조사 응답 데이터 검증 및 표준화

In [1]:
# 필요한 라이브러리 import
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Dict, Optional, Any
import enum
from datetime import datetime
from pprint import pprint

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash-lite')

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {gemini_model}")

# 클라이언트 초기화
client = genai.Client(api_key=api_key)

API 키 설정 확인: ✓
모델 확인: gemini-2.5-flash-lite


### 1. 고객 문의 데이터 전처리

#### LLM 응답 스키마 설계 가이드

**스키마 설계 프로세스**

**1. 비즈니스 질문부터 시작**

Q: LLM으로부터 무엇을 얻고 싶은가?\
A: 고객 문의를 자동 분류하고, 우선순위를 정하고, 감정을 파악하고 싶다

**2. 필요한 정보 나열**

- 문의 유형: 환불? 배송? 상품?
- 긴급도: 낮음/보통/높음/긴급
- 고객 감정: 긍정/중립/부정
- 핵심 키워드: ["배송 지연", "환불 요청"]
- 주문번호: ORDER-12345 (있을 수도, 없을 수도)
- 권장 응답 가이드 : "배송 지연 사과 후 예상 도착일 안내 필요"

**3. 데이터 타입 결정**

| 정보 | 타입 | 이유 |
|------|------|------|
| 문의 유형 | *Enum* | 고정된 카테고리 |
| 긴급도 | *Enum* | 4단계 중 선택 |
| 감정 | *Enum* | 3가지 중 선택 |
| 키워드 | *List[str]* | 여러 개 가능 |
| 주문번호 | *Optional[str]* | 없을 수도 있음 |

**핵심 결정 기준**

- *Enum*을 쓰는 경우: 정해진 선택지가 있을 때 (문의 유형, 긴급도, 감정)
- *List*를 쓰는 경우: 여러 개일 수 있을 때 (키워드 여러 개)
- *Optional*을 쓰는 경우: 없을 수도 있을 때 (주문번호, 연락처)

**핵심 메시지**

스키마 = "내가 원하는 답의 형태"

- 스키마 없으면: "이 문의는 환불 관련이고 급하며 고객이 화났습니다..." → 파싱 불가능
- 스키마 있으면: {"category": "refund", "urgency": "urgent", "emotion": "angry"} → 바로 사용 가능

*구조화 = 분석의 시작점*

In [3]:
# =============================================================================
#  고객 문의 데이터 전처리 스키마 정의
# =============================================================================

class InquiryType(str, enum.Enum):
    REFUND = "refund"           # 환불 문의
    EXCHANGE = "exchange"       # 교환 문의
    DELIVERY = "delivery"       # 배송 문의
    PRODUCT = "product"         # 상품 문의
    TECHNICAL = "technical"     # 기술 지원
    COMPLAINT = "complaint"     # 불만 사항
    OTHER = "other"            # 기타

class UrgencyLevel(str, enum.Enum):
    LOW = "low"                # 낮음
    MEDIUM = "medium"          # 보통
    HIGH = "high"              # 높음
    URGENT = "urgent"          # 긴급

class EmotionType(str, enum.Enum):
    POSITIVE = "positive"      # 긍정적
    NEUTRAL = "neutral"        # 중립적
    NEGATIVE = "negative"      # 부정적

class CustomerInquiry(BaseModel):
    original_text: str = Field(description="원본 텍스트")
    category: InquiryType = Field(description="문의 유형 분류")
    urgency: UrgencyLevel = Field(description="긴급도 평가")
    emotion: EmotionType = Field(description="고객 감정 상태")
    key_issues: List[str] = Field(description="주요 이슈 키워드", max_items=5)
    order_number: Optional[str] = Field(default=None, description="주문번호")
    suggested_response: str = Field(description="권장 응답 방향", max_length=200)

print("✅ 고객 문의 분석 스키마 정의 완료")

✅ 고객 문의 분석 스키마 정의 완료


/var/folders/mt/b5bzczgn14s85rhfsvnlr33h0000gn/T/ipykernel_42632/2843491637.py:30: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  key_issues: List[str] = Field(description="주요 이슈 키워드", max_items=5)


In [4]:
CustomerInquiry.model_json_schema()

{'$defs': {'EmotionType': {'enum': ['positive', 'neutral', 'negative'],
   'title': 'EmotionType',
   'type': 'string'},
  'InquiryType': {'enum': ['refund',
    'exchange',
    'delivery',
    'product',
    'technical',
    'complaint',
    'other'],
   'title': 'InquiryType',
   'type': 'string'},
  'UrgencyLevel': {'enum': ['low', 'medium', 'high', 'urgent'],
   'title': 'UrgencyLevel',
   'type': 'string'}},
 'properties': {'original_text': {'description': '원본 텍스트',
   'title': 'Original Text',
   'type': 'string'},
  'category': {'$ref': '#/$defs/InquiryType', 'description': '문의 유형 분류'},
  'urgency': {'$ref': '#/$defs/UrgencyLevel', 'description': '긴급도 평가'},
  'emotion': {'$ref': '#/$defs/EmotionType', 'description': '고객 감정 상태'},
  'key_issues': {'description': '주요 이슈 키워드',
   'items': {'type': 'string'},
   'maxItems': 5,
   'title': 'Key Issues',
   'type': 'array'},
  'order_number': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': '주문번호'

In [ ]:
# =============================================================================
#  샘플 고객 문의 데이터
# =============================================================================

raw_inquiries = [
    "주문번호 ORDER-230415입니다. 환불 어떻게 하나요??? 급합니다ㅠㅠ 배송비 차감되나요??",
    "배송 언제 오나요...... 일주일 지났는데 추적도 안돼요ㅠㅠ 빨리 답변 부탁드립니다",
    "상품 불량입니다. 교환 가능한가요? 포장 찢어져서 왔고 제품에 스크래치 있어요. 010-1234-5678",
    "로그인이 안 됩니다... 비밀번호 재설정 눌러도 이메일 안 와요 도와주세요ㅠ",
    "감사합니다! 빠른 배송 덕분에 생일 선물 제때 받았어요 ^^ 포장도 깔끔하네요",
    "주문번호 ORDER-230512 / 노트북 파우치 색상 변경 가능할까요? 블랙 → 네이비로요",
]

# =============================================================================
#  시스템 프롬프트 정의
# :AI에게 내리는 출력 지침 
# =============================================================================

system_instruction = """
당신은 고객 서비스 데이터 분석 전문가입니다.
고객 문의를 받아 구조화된 형태로 분석하세요.

[분석 항목]

1. 문의 유형 (category)
   - refund: 환불 요청 관련
   - exchange: 교환 요청 관련
   - delivery: 배송 조회, 지연, 분실 관련
   - product: 상품 정보, 재고, 사양 문의
   - technical: 로그인, 결제, 시스템 오류 등 기술 문제
   - complaint: 서비스 불만, 항의
   - other: 위 항목에 해당하지 않는 경우

2. 긴급도 (urgency)
   - urgent: 즉시 처리 요구
   ->  언제, 어떤 경우에 Urgent 설명을 구체적으로
   ->  예시를 들어줘도 좋음
   - high: 시간 압박 표현
   - medium: 일반적인 답변 요청
   - low: 단순 문의, 긴급 표현 없음

3. 감정 (emotion)
   - positive: 감사, 만족, 칭찬 표현
   - neutral: 담담한 문의, 정보 요청
   - negative: 불만, 실망, 화남 표현 

4. 핵심 이슈 (key_issues)
   - 문의의 핵심 문제를 명사형 키워드로 2-5개 추출
   - 예: ["환불 절차", "배송비 정책"], ["배송 지연", "추적 불가"]

5. 주문번호 (order_number)
   - "주문번호", "ORDER-", 숫자 조합에서 추출
   - 형식: ORDER-XXXXXX 또는 XXXXXX-XXX
   - 없으면 null

6. 응답 가이드 (suggested_response)
   - 상담원이 우선적으로 안내해야 할 내용을 1-2문장으로 제안
   - 예: "환불 절차를 안내하고 배송비 차감 여부를 명확히 설명"
   - 200자 이내로 작성

한국어 문의의 뉘앙스(존댓말, 이모티콘, 중복 표현)를 정확히 파악하세요.
"""

# =============================================================================
#  고객 문의 데이터 전처리 실행
# =============================================================================

print("🔄 고객 문의 데이터 전처리 시작...")
print("=" * 80)

# 처리된 문의 데이터를 저장할 리스트
processed_inquiries = []

# 각 고객 문의를 순회하며 LLM으로 분석
for i, inquiry in enumerate(raw_inquiries, 1):
    print(f"\n📝 문의 {i}: {inquiry[:60]}...")
    
    # LLM에 전달할 프롬프트 구성
    prompt = f"다음 고객 문의를 분석해주세요:\n\n{inquiry}"
    
    try:
        # LLM 생성 설정
        generation_config = types.GenerateContentConfig(
            temperature=0.2,                          # 창의성(0~2) 낮은 temperature로 일관된 분류 결과 유도
            response_mime_type='application/json',    # JSON 형식으로 응답 받기
            response_schema=CustomerInquiry,          # Pydantic 스키마로 구조화된 응답 강제
            system_instruction=system_instruction     # 시스템 프롬프트 적용
        )
        
        # Gemini API 호출
        response = client.models.generate_content(
            model=gemini_model,
            contents=prompt,
            config=generation_config
        )
        
        # 파싱된 결과 가져오기 (CustomerInquiry 객체로 자동 변환됨)
        result = response.parsed
        
        # 결과가 정상적으로 반환된 경우
        if result:
            # 딕셔너리 형태로 변환하여 리스트에 저장
            processed_inquiries.append(
                result.model_dump()
            )
            
            # 분석 결과 출력
            print(f"   유형: {result.category.value} | 긴급도: {result.urgency.value} | 감정: {result.emotion.value}")
            print(f"   이슈: {', '.join(result.key_issues)}")
            if result.order_number:  # 주문번호가 있는 경우만 출력
                print(f"   주문: {result.order_number}")
            print(f"   응답가이드: {result.suggested_response}")
            
    except Exception as e:
        # API 호출 실패 또는 파싱 오류 처리
        print(f"❌ 처리 오류: {e}")

# 전체 처리 결과 요약
print(f"\n{'=' * 80}")
print(f"✅ 총 {len(processed_inquiries)}개 문의 처리 완료")

In [ ]:
# 원본 응답
pprint(response.parsed)
pprint(response.parsed.model_dump())

In [ ]:
# json으로 저장
pprint(processed_inquiries)

### 2. 개인정보 마스킹 및 익명화

**목적**

텍스트 데이터에서 개인정보(PII)를 자동으로 탐지하고 안전하게 마스킹 처리

**PII (Personally Identifiable Information)란?**

개인을 식별할 수 있는 정보
- 이름, 전화번호, 이메일, 주소
- 주민등록번호, 신용카드 번호, 계좌번호

**왜 필요한가?**

- 개인정보보호법 준수
- 데이터 분석 시 개인정보 유출 방지
- AI 학습 데이터 안전성 확보
- 고객 신뢰 및 법적 리스크 관리

**실무 활용 사례**
- 고객 문의 데이터 분석 전 익명화
- 로그 데이터 저장 시 개인정보 제거
- 데이터 공유/외주 시 민감정보 마스킹

### 개인정보 마스킹 스키마 설계 가이드

**설계 프로세스**

**1. 비즈니스 질문부터 시작**

Q: LLM으로부터 무엇을 얻고 싶은가?\
A: 텍스트에서 개인정보를 찾아내고, 안전하게 마스킹 처리하고, 위험도를 평가하고 싶다

**2. 필요한 정보 나열**

*개별 개인정보 항목마다 필요한 정보:*
- 원본 값: "010-1234-5678"
- 정보 타입: phone (이름/전화번호/이메일/주민번호/카드번호/계좌번호/주소)
- 탐지 신뢰도: 0.95 (0.0~1.0, 얼마나 확실한가?)
- 마스킹된 값: "010-****-5678"

*전체 텍스트에 대한 정보:*
- 원본 텍스트: "안녕하세요, 제 이름은 김철수이고 연락처는 010-1234-5678입니다."
- 익명화된 텍스트: "안녕하세요, 제 이름은 김**이고 연락처는 010-****-5678입니다."
- 탐지된 개인정보 개수: 2개
- 전체 위험도: low/medium/high/critical

**3. 데이터 타입 결정**

| 정보 | 타입 | 이유 |
|------|------|------|
| 개인정보 타입 | *Enum* | 7가지 고정 카테고리 |
| 위험도 | *Enum* | 4단계 중 선택 |
| 신뢰도 | *float* | 0.0~1.0 연속값 |
| 탐지된 PII 목록 | *List[PIIItem]* | 여러 개 가능, 중첩 구조 |
| 원본/마스킹 값 | *str* | 문자열 |
| 권장사항 | *List[str]* | 여러 개 가능 |

**4. 중첩 구조 설계**

개별 PII 항목 (PIIItem):
- value: "010-1234-5678"
- pii_type: "phone"
- confidence: 0.95
- masked_value: "010-****-5678"

전체 분석 결과 (PIIDetection):
- detected_pii: [PIIItem 1, PIIItem 2, ...]
- total_pii_count: 3
- risk_level: "high"

**핵심 결정 기준**

- **Enum**을 쓰는 경우: 정해진 선택지 (PII 타입, 위험도)
- **List[타입]**을 쓰는 경우: 같은 구조의 항목이 여러 개 (탐지된 개인정보 목록)
- **float**를 쓰는 경우: 연속적인 수치 (신뢰도 0.0~1.0)
- **중첩 구조**를 쓰는 경우: 복잡한 데이터 관계 표현 (각 PII마다 상세 정보 필요)

중첩 스키마 = "복잡한 데이터를 체계적으로 구조화"
- 평면 구조: {"pii1": "전화번호", "pii2": "이메일"} → 확장 어려움
- 중첩 구조: {"detected_pii": [{"value": "010-1234-5678", "type": "phone", "confidence": 0.95}]} → 유연하고 확장 가능

In [ ]:
# =============================================================================
#  개인정보 마스킹 스키마 정의 (중첩 구조)
# =============================================================================

class PIIType(str, enum.Enum):
    NAME = "name"                    # 이름
    PHONE = "phone"                  # 전화번호
    EMAIL = "email"                  # 이메일
    ADDRESS = "address"              # 주소
    ID_NUMBER = "id_number"          # 주민등록번호
    CREDIT_CARD = "credit_card"      # 신용카드 번호
    ACCOUNT = "account"              # 계좌번호

class RiskLevel(str, enum.Enum):
    LOW = "low"                      # 낮음
    MEDIUM = "medium"                # 보통
    HIGH = "high"                    # 높음
    CRITICAL = "critical"            # 매우 높음

# 개별 PII 항목 정보
class PIIItem(BaseModel):
    """개별 개인정보 항목 (중첩 구조)"""
    original_value: str = Field(description="탐지된 원본 개인정보 값")
    pii_type: PIIType = Field(description="개인정보 유형")
    confidence_score: float = Field(ge=0.0, le=1.0, description="탐지 신뢰도 (0.0~1.0)")
    masked_value: str = Field(description="마스킹 처리된 값")

# 메인 스키마 (중첩 구조 활용)
class PIIDetectionResult(BaseModel):
    """개인정보 탐지 및 마스킹 결과"""
    
    # 원본 및 처리 결과
    original_text: str = Field(description="원본 텍스트")
    anonymized_text: str = Field(description="개인정보가 마스킹된 텍스트")
    
    # 탐지된 개인정보 목록 (중첩 구조)
    detected_items: List[PIIItem] = Field(description="탐지된 개인정보 항목 리스트")
    
    # 전체 요약 정보
    total_count: int = Field(ge=0, description="탐지된 개인정보 총 개수")
    risk_level: RiskLevel = Field(description="전체 위험도 평가")


print("✅ 중첩 구조 개인정보 마스킹 스키마 정의 완료")

In [ ]:
# =============================================================================
#  샘플 개인정보 포함 텍스트 데이터
# =============================================================================

pii_samples = [
    "안녕하세요, 홍길동입니다. 010-1234-5678로 연락주세요. 이메일은 hong@gmail.com이고 주소는 서울시 강남구 역삼동 123-45입니다.",
    "주문자: 김영희, 전화: 02-9876-5432, 카드번호: 1234-5678-9012-3456, 배송지: 부산시 해운대구 센텀로 99",
    "고객정보 - 성명: 박철수, 주민번호: 801225-1234567, 계좌: 국민은행 123-456-789012",
    "일반적인 상품 문의입니다. 재료와 사이즈에 대해 알고 싶어요."
    "회원가입 정보: 이메일 lee@naver.com, 휴대폰 010-9999-8888, 생년월일 1990.03.15",
]

# =============================================================================
#  시스템 프롬프트 정의
# =============================================================================

system_instruction = """
당신은 개인정보보호 전문가입니다.
텍스트에서 개인정보를 탐지하고 안전하게 마스킹 처리하세요.

[분석 항목]

1. 개인정보 탐지 (detected_items)
   각 개인정보 항목마다 다음을 포함:
   
   - original_value: 탐지된 원본 값
     예: "010-1234-5678", "홍길동", "hong@gmail.com"
   
   - pii_type: 개인정보 유형
     * name: 한글/영문 이름
     * phone: 전화번호 (010-XXXX-XXXX, 02-XXX-XXXX 등)
     * email: 이메일 주소
     * address: 주소 (도로명, 지번, 동/호수)
     * id_number: 주민등록번호 (XXXXXX-XXXXXXX)
     * credit_card: 신용카드 번호 (XXXX-XXXX-XXXX-XXXX)
     * account: 계좌번호 (은행명 + 번호)
   
   - confidence_score: 탐지 신뢰도 (0.0~1.0)
     * 0.9~1.0: 명확한 개인정보 패턴 (전화번호 형식, 이메일 @ 포함)
     * 0.7~0.9: 높은 확률 (한글 이름 2-4자, 주소 키워드)
     * 0.5~0.7: 가능성 있음 (불확실한 패턴)
   
   - masked_value: 마스킹 처리 결과
     * 이름: 홍** (첫 글자만 보존)
     * 전화번호: 010-****-5678 (뒤 4자리만 보존)
     * 이메일: h***@gmail.com (첫 글자 + 도메인만 보존)
     * 주소: 서울시 강남구 [상세주소 비공개] (시/구까지만, 동/번지 완전 마스킹)
     * 주민번호: 801225-1****** (앞 6자리만 보존)
     * 카드번호: 1234-****-****-3456 (앞/뒤 4자리만 보존)
     * 계좌번호: 국민은행 123-***-****** (은행명 + 앞자리만)

2. 익명화된 텍스트 (anonymized_text)
   - 원본 텍스트에서 개인정보를 마스킹 값으로 대체
   - 예: "안녕하세요, 홍**입니다. 010-****-5678로 연락주세요."
   - 주소 예시: "서울시 강남구 [상세주소 비공개]"

3. 전체 요약 (total_count, risk_level)
   - total_count: 탐지된 개인정보 총 개수
   - risk_level: 위험도 평가
     * low: 개인정보 없음 또는 일반 정보만
     * medium: 이름, 전화번호, 이메일 등 일반 연락처
     * high: 이름+연락처 조합 또는 주소 포함
     * critical: 주민번호, 카드번호, 계좌번호 등 민감정보 포함

한국어 개인정보 패턴을 정확히 인식하고 개인정보보호법을 준수하세요.
주소는 반드시 시/구 수준까지만 공개하고 상세 주소는 완전히 마스킹하세요.
"""

# =============================================================================
#  개인정보 탐지 및 마스킹 함수
# =============================================================================

def detect_and_mask_pii(text: str) -> PIIDetectionResult:
    """
    텍스트에서 개인정보를 탐지하고 마스킹 처리
    
    Args:
        text: 개인정보가 포함될 수 있는 원본 텍스트
        
    Returns:
        PIIDetectionResult: 탐지 및 마스킹 결과가 담긴 구조화된 객체
    """
    
    # LLM에 전달할 프롬프트 구성
    prompt = f"다음 텍스트에서 개인정보를 탐지하고 마스킹해주세요:\n\n{text}"
    
    try:
        # LLM 생성 설정
        generation_config = types.GenerateContentConfig(
            temperature=0.1,                           # 매우 낮은 temperature로 일관된 탐지
            response_mime_type='application/json',     # JSON 형식 응답
            response_schema=PIIDetectionResult,        # Pydantic 스키마로 구조화
            system_instruction=system_instruction      # 시스템 프롬프트 적용
        )
        
        # Gemini API 호출
        response = client.models.generate_content(
            model=gemini_model,
            contents=prompt,
            config=generation_config
        )
        
        # 파싱된 결과 반환 (PIIDetectionResult 객체)
        return response.parsed
        
    except Exception as e:
        # API 호출 실패 또는 파싱 오류 처리
        print(f"❌ 처리 오류: {e}")
        return None

# =============================================================================
#  개인정보 마스킹 처리 실행
# =============================================================================

print("🔒 개인정보 마스킹 처리 시작...")
print("=" * 80)

# 처리 결과를 저장할 리스트
pii_results = []

# 각 샘플 텍스트를 순회하며 개인정보 탐지 및 마스킹
for i, sample in enumerate(pii_samples, 1):
    print(f"\n📝 샘플 {i}: {sample[:60]}...")
    
    # 함수 호출로 개인정보 탐지 및 마스킹
    result = detect_and_mask_pii(sample)
    
    if result:
        # 딕셔너리 형태로 변환하여 저장
        pii_results.append(result.model_dump())
        
        # 처리 결과 출력
        print(f"   익명화: {result.anonymized_text[:60]}...")
        print(f"   위험도: {result.risk_level.value} | 탐지 개수: {result.total_count}개")
        
        # 탐지된 개인정보 상세 출력
        if result.detected_items:
            print(f"   탐지 정보:")
            for item in result.detected_items:
                print(f"     - {item.pii_type.value}: {item.original_value} → {item.masked_value} (신뢰도: {item.confidence_score:.2f})")

# 전체 처리 결과 요약
print(f"\n{'=' * 80}")
print(f"✅ 총 {len(pii_results)}개 텍스트 익명화 완료")





# input은 5개인데 output이 더 적게 나오는 경우는 왜그럴까?
# 두 가지 문제

# 1. 무료 API 호출한도 1분에 15개 개수 제한 --> 너무나 빠르게 요청을 많이 하면 응답 X --> 응답마다 딜레이주기
# 2. 응답구조 스키마를 설계 했지만 스키마대로 답변을 못한 경우 --> 스키마 재설정(제약조건 다시 고려) | (프롬프트를 더 명확하게 강조해서 작성)

In [ ]:
pprint(pii_results)

### 3. 설문조사 데이터 검증 및 표준화

목적
비정형 설문조사 응답을 구조화된 데이터로 변환하고 품질 검증

왜 필요한가?
- 설문조사 응답은 자유 형식이 많아 일관성 부족
- 나이 표현: "25살", "스물다섯", "20대 중반" 등 다양
- 소득 표현: "3천", "3000만원", "연봉 30m" 등 비표준
- 분석을 위해서는 표준화된 형태 필요

실무 활용 사례
- 온라인 설문조사 응답 자동 정제
- 데이터 분석을 위한 카테고리화 (20대, 30대 등)
- 불량 데이터 자동 필터링

In [ ]:
# =============================================================================
# 설문조사 검증 스키마
# =============================================================================

# 데이터 품질 등급 정의 (4단계 평가)
class DataQuality(str, enum.Enum):
    """설문조사 응답 데이터의 품질을 4단계로 평가"""
    EXCELLENT = "excellent"      # 우수: 모든 정보가 명확하고 일관됨
    GOOD = "good"               # 양호: 대부분 정보가 양호하나 일부 불분명
    FAIR = "fair"               # 보통: 절반 정도 정보만 신뢰할 수 있음
    POOR = "poor"               # 불량: 대부분 정보가 불분명하거나 모순됨

# 성별 타입 정의
class GenderType(str, enum.Enum):
    """응답자 성별 (3가지 고정 선택지)"""
    MALE = "male"               # 남성
    FEMALE = "female"           # 여성
    UNKNOWN = "unknown"         # 불명 (정보 없음 또는 응답 거부)

# 표준화/카테고리화 정보 (중첩 구조)
class StandardizedInfo(BaseModel):
    """
    분석을 위해 원본 데이터를 카테고리화한 표준화 정보
    - 원본: age=25 → 표준화: age_group="20대"
    - 원본: occupation="회사원" → 표준화: job_category="사무직"
    """
    age_group: Optional[str] = Field(
        description="연령대 카테고리 (예: 10대, 20대, 30대, 40대)"
    )
    job_category: Optional[str] = Field(
        description="직업 대분류 (예: 사무직, 전문직, 서비스직, 자영업)"
    )
    income_range: Optional[str] = Field(
        description="소득 구간 (예: 3000만원 이하, 3000-5000만원, 5000만원 이상)"
    )
    education_level: Optional[str] = Field(
        description="학력 표준화 (예: 고졸, 대졸, 대학원졸)"
    )
    region: Optional[str] = Field(
        description="지역 표준화 (예: 서울, 경기, 광역시, 지방)"
    )

# 메인 스키마: 설문조사 검증 결과
class SurveyValidation(BaseModel):
    """
    설문조사 응답을 검증하고 표준화한 최종 결과
    - 원본 정보 추출 (age, gender, occupation 등)
    - 표준화 정보 생성 (age_group, job_category 등)
    - 데이터 품질 평가 (quality, confidence, issues)
    """
    
    # 원본 응답 텍스트
    original_response: str = Field(
        description="사용자가 입력한 원본 설문조사 응답 텍스트"
    )
    
    # =========================================================================
    # 추출된 원본 정보 (응답자가 입력한 그대로)
    # =========================================================================
    age: Optional[int] = Field(
        ge=0, le=150, 
        description="나이 (숫자로 변환, 예: '스물다섯살' → 25)"
    )
    gender: GenderType = Field(
        description="성별 (male/female/unknown 중 하나)"
    )
    occupation: Optional[str] = Field(
        description="직업 (원본 표현 그대로, 예: '회사원', '개발자')"
    )
    annual_income: Optional[int] = Field(
        ge=0, 
        description="연소득 만원 단위 (예: '3000만원' → 3000)"
    )
    education: Optional[str] = Field(
        description="학력 (원본 표현 그대로, 예: '대졸', '석사')"
    )
    location: Optional[str] = Field(
        description="거주지역 (원본 표현 그대로, 예: '서울', '경기도')"
    )
    
    # =========================================================================
    # 데이터 분석을 위한 표준화 정보 (중첩 구조)
    # =========================================================================
    standardized: StandardizedInfo = Field(
        description="원본 정보를 분석 가능한 카테고리로 변환한 표준화 데이터"
    )
    
    # =========================================================================
    # 데이터 품질 검증 결과
    # =========================================================================
    data_quality: DataQuality = Field(
        description="응답 데이터의 전체 품질 등급 (excellent/good/fair/poor)"
    )
    confidence_score: float = Field(
        ge=0.0, le=1.0, 
        description="추출 결과에 대한 신뢰도 점수 (0.0~1.0, 높을수록 신뢰)"
    )
    issues_found: List[str] = Field(
        description="발견된 문제점 목록 (예: ['나이 정보 누락', '소득 표현 모호'])",
        max_items=5  # 최대 5개까지만
    )

print("✅ 설문조사 검증 스키마 정의 완료")

In [ ]:
# =============================================================================
# 샘플 설문조사 응답 데이터 (비정형 형태)
# =============================================================================

# 실제 설문조사에서 수집되는 다양한 형식의 응답 예시
# - 한글/영어 혼용, 띄어쓰기 불규칙, 단위 표현 다양, 구어체 표현 포함
survey_responses = [
    "나이: 스물다섯살, 성별: 남자, 직업: 회사원, 연봉: 3000만원정도, 학력: 대졸, 지역: 서울",
    "25세 여성 개발자입니다. 연소득 3천5백. 석사과정 중이고 경기도 거주",
    "thirty years old, male, teacher, 2800만원, 학사졸업, 부산시",
    "나이 모름, 직업: 프리랜서, 돈: 많이벌어요 ㅋㅋ, 서울 강남 살아요",
    "28살 여자 간호사 연봉 3200 대학졸업 대구",
    "19세, 학생, 무직, 용돈 받아서 살아요, 부모님과 거주 인천",
    "40대 초반 남성 자영업자 연수입 5000만원 고졸 충청도"
]

# =============================================================================
# 시스템 프롬프트 정의
# =============================================================================

system_instruction = """
당신은 설문조사 데이터 품질 관리 전문가입니다.
비정형 텍스트 응답을 구조화된 데이터로 변환하고 품질을 평가하세요.

[원본 정보 추출 규칙]

1. 나이 (age)
   - 정확한 숫자만 추출: "25살" → 25, "서른" → 30
   - 범위 표현은 null: "20대 후반", "40대 초반" → null

2. 성별 (gender)
   - male: "남자", "남성", "male"
   - female: "여자", "여성", "female"
   - unknown: 정보 없음

3. 직업 (occupation)
   - 원본 표현 그대로 추출
   - 무직/불명확 → null

4. 연소득 (annual_income)
   - 만원 단위 숫자로 변환: "3000만원" → 3000, "3천5백" → 3500
   - 불명확 → null

5. 학력 (education)
   - 원본 표현 그대로

6. 거주지역 (location)
   - 시/도 수준만: "서울", "경기도", "부산시"

[표준화 규칙 - standardized 필드]

1. 연령대 (age_group): "10대", "20대", "30대", "40대", "50대", "60대 이상"

2. 직업 대분류 (job_category)
   - 사무직: 회사원, 직장인
   - 전문직: 의사, 교사, 개발자, 간호사
   - 자영업: 자영업자
   - 학생: 학생
   - 무직: 무직
   - 기타: 프리랜서 등

3. 소득 구간 (income_range)
   - "3000만원 미만", "3000-5000만원", "5000-7000만원", "7000만원 이상"

4. 학력 수준 (education_level)
   - "고졸 이하", "대졸", "대학원졸"

5. 지역 (region)
   - "서울", "경기", "광역시" (부산/대구/인천 등), "지방" (충청/경상/전라/강원/제주)

[품질 평가]

- data_quality
  * excellent: 5개 이상 정보 명확
  * good: 4-5개 정보 신뢰 가능
  * fair: 2-3개 정보만 신뢰
  * poor: 1개 이하

- confidence_score: 0.0~1.0 (높을수록 신뢰)

- issues_found: 발견된 문제점 (최대 5개)
  예: "나이 정보 없음", "소득 표현 불명확", "구어체 과다"

불확실한 정보는 null로 처리하세요.
"""

# =============================================================================
# 설문조사 응답 검증 및 표준화 함수
# =============================================================================

def validate_survey_response(response: str) -> SurveyValidation:
    """
    비정형 설문조사 응답을 구조화된 데이터로 변환하고 품질 검증
    
    Args:
        response: 설문조사 원본 응답 텍스트
        
    Returns:
        SurveyValidation: 검증 및 표준화된 결과 객체
        - 원본 정보 추출 (age, gender, occupation 등)
        - 표준화 정보 (age_group, job_category 등)
        - 품질 평가 (data_quality, confidence_score, issues_found)
    """
    
    # LLM에 전달할 프롬프트 구성
    prompt = f"다음 설문조사 응답을 검증하고 표준화해주세요:\n\n{response}"
    
    try:
        # LLM 생성 설정
        generation_config = types.GenerateContentConfig(
            temperature=0.1,                        # 낮은 temperature로 일관된 추출
            response_mime_type='application/json',  # JSON 형식 응답
            response_schema=SurveyValidation,       # Pydantic 스키마로 구조화
            system_instruction=system_instruction   # 시스템 프롬프트 적용
        )
        
        # Gemini API 호출
        response_obj = client.models.generate_content(
            model=gemini_model,
            contents=prompt,
            config=generation_config
        )
        
        # 파싱된 결과 반환 (SurveyValidation 객체)
        return response_obj.parsed
        
    except Exception as e:
        # API 호출 실패 또는 파싱 오류 처리
        print(f"❌ 설문조사 검증 오류: {e}")
        return None

# =============================================================================
# 설문조사 검증 실행
# =============================================================================

print("📊 설문조사 데이터 검증 시작...")
print("=" * 80)

# 검증 결과를 저장할 리스트
validated_surveys = []

# 각 설문조사 응답을 순회하며 검증 및 표준화
for i, response in enumerate(survey_responses, 1):
    print(f"\n📝 응답 {i}: \"{response}\"")
    
    # 함수 호출로 응답 검증
    result = validate_survey_response(response)
    
    if result:
        # Pydantic 모델을 딕셔너리로 변환하여 저장
        validated_surveys.append(
            result.model_dump()
        )
        
        # 검증 결과 출력: 원본 정보 vs 표준화 정보 비교
        print(f"  - 원본정보: 나이={result.age}, 성별={result.gender.value}, 직업={result.occupation}")
        print(f"  - 표준화: {result.standardized.age_group}, {result.standardized.job_category}, {result.standardized.income_range}")
        print(f"  - 품질: {result.data_quality.value} (신뢰도: {result.confidence_score:.2f})")
        
        # 이슈가 발견된 경우에만 출력
        if result.issues_found:
            print(f"  ⚠️ 이슈: {', '.join(result.issues_found)}")

# 전체 처리 결과 요약
print(f"\n{'=' * 80}")
print(f"✅ 총 {len(validated_surveys)}개 응답 검증 완료")

In [ ]:
pprint(validated_surveys)